# Hyperbolic Geometry: Agent Initialization

Explore agent generation on the hyperbolic plane $\mathbb{H}^2$ using the native disk model.

Following Krioukov et al. (2010), Section IV.A:
- Angular coordinate $\theta \sim \text{Uniform}(0, 2\pi)$ — similarity dimension
- Radial coordinate $r$ with density $\rho(r) = \frac{\sinh r}{\cosh R - 1}$ — popularity dimension
- Connection probability based on hyperbolic distance

**Key questions:**
- Does this initialization naturally produce heavy-tailed degree distributions?
- How does the resulting network structure compare to the torus baseline?
- What do structural hole metrics look like at t=0?

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 120

RNG = np.random.default_rng(42)

## 1. Hyperbolic Disk Geometry

### Distance (Eq. 5 in Krioukov, with curvature $K = -\zeta^2$, $\zeta = 1$)

$$\cosh x = \cosh r \cosh r' - \sinh r \sinh r' \cos \Delta\theta$$

where $\Delta\theta = \pi - |\pi - |\theta - \theta'||$.

### Approximate distance for large $r, r'$ (Eq. 6)

$$x \approx r + r' + 2 \ln \sin \frac{\Delta\theta}{2}$$

### Node density (Eq. 7)

$$\rho(r) = \frac{\sinh r}{\cosh R - 1} \approx e^{r - R}$$

### Disk radius

From Eq. 13, $\bar{k} \approx \frac{8}{\pi} N e^{-R/2}$, so:

$$R \approx 2 \ln \frac{8N}{\pi \bar{k}}$$

In [ ]:
def hyperbolic_distance(r1: float, theta1: float, r2: float, theta2: float) -> float:
    """Exact hyperbolic distance between two points (Eq. 5, zeta=1)."""
    delta_theta = np.pi - np.abs(np.pi - np.abs(theta1 - theta2))
    cosh_x = (
        np.cosh(r1) * np.cosh(r2)
        - np.sinh(r1) * np.sinh(r2) * np.cos(delta_theta)
    )
    # Clamp for numerical stability
    cosh_x = np.maximum(cosh_x, 1.0)
    return np.arccosh(cosh_x)


def hyperbolic_distance_matrix(r: np.ndarray, theta: np.ndarray) -> np.ndarray:
    """Compute pairwise hyperbolic distances (vectorized)."""
    n = len(r)
    r1 = r[:, np.newaxis]
    r2 = r[np.newaxis, :]
    t1 = theta[:, np.newaxis]
    t2 = theta[np.newaxis, :]
    
    delta_theta = np.pi - np.abs(np.pi - np.abs(t1 - t2))
    cosh_x = np.cosh(r1) * np.cosh(r2) - np.sinh(r1) * np.sinh(r2) * np.cos(delta_theta)
    cosh_x = np.maximum(cosh_x, 1.0)
    return np.arccosh(cosh_x)

## 2. Agent Generation: Generalized Krioukov Prescription

The generalized density uses parameter $\alpha$ to control how agents spread across the disk:

$$\rho(r) = \frac{\alpha \sinh(\alpha r)}{\cosh(\alpha R) - 1}$$

- $\alpha = 1$: standard Krioukov (almost all agents at boundary, power-law exponent $\gamma = 3$)
- $\alpha = 0.5$: more spread toward center ($\gamma = 2$)
- Lower $\alpha$ = more central agents, heavier-tailed degree distribution

Inverse CDF sampling:
$$r = \frac{1}{\alpha} \operatorname{arccosh}\left(1 + u(\cosh(\alpha R) - 1)\right), \quad u \sim \text{Uniform}(0, 1)$$

In [ ]:
def compute_disk_radius(n: int, target_mean_degree: float) -> float:
    """Compute disk radius R from Eq. 13: k_bar ~ (8/pi) * N * exp(-R/2)."""
    return 2.0 * np.log(8.0 * n / (np.pi * target_mean_degree))


def generate_hyperbolic_uniform(
    n: int,
    R: float,
    rng: np.random.Generator,
    alpha: float = 1.0,
) -> tuple[np.ndarray, np.ndarray]:
    """Generate n agents on the hyperbolic disk (generalized Krioukov, Eq. 17).
    
    alpha controls radial spread (Eq. 29: gamma = 2*alpha + 1 for zeta=1):
      - alpha=1.0: standard Krioukov, gamma=3, most agents at boundary
      - alpha=0.5: gamma=2, more spread toward center, heavier-tailed degrees
    
    NOTE: alpha is a key tunable parameter. Adjust based on the target
    power-law exponent of the network being modeled.
    
    Returns (r, theta).
    """
    theta = rng.uniform(0, 2 * np.pi, size=n)
    
    # Inverse CDF: r = (1/alpha) * arccosh(1 + u * (cosh(alpha*R) - 1))
    u = rng.uniform(0, 1, size=n)
    r = np.arccosh(1.0 + u * (np.cosh(alpha * R) - 1.0)) / alpha
    
    return r, theta


def poincare_coords(r, theta, R, stretch=0.6):
    """Map hyperbolic (r, theta) to Poincare disk (x, y) with visual scaling.
    
    The true Poincare mapping tanh(r/2) compresses everything near the boundary.
    We use (r/R)^stretch instead for readability:
      - stretch=1.0: linear in hyperbolic radius
      - stretch=0.5: square-root, spreads center out more
      - stretch=0.6: reasonable default
    """
    r_visual = (r / R) ** stretch
    x = r_visual * np.cos(theta)
    y = r_visual * np.sin(theta)
    return x, y

In [ ]:
N = 300
TARGET_K = 10
ALPHA = 0.5  # Tunable: controls power-law exponent gamma = 2*alpha + 1
R = compute_disk_radius(N, TARGET_K)
print(f'N = {N}, target mean degree = {TARGET_K}, disk radius R = {R:.3f}, alpha = {ALPHA} (gamma = {2*ALPHA+1})')

r, theta = generate_hyperbolic_uniform(N, R, RNG, alpha=ALPHA)
x_vis, y_vis = poincare_coords(r, theta, R, stretch=0.6)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Poincare disk view (visual scaling)
circle = plt.Circle((0, 0), 1, fill=False, color='black', linewidth=1.5)
axes[0].add_patch(circle)
axes[0].scatter(x_vis, y_vis, s=12, alpha=0.6, c=r, cmap='viridis')
axes[0].set_xlim(-1.15, 1.15)
axes[0].set_ylim(-1.15, 1.15)
axes[0].set_aspect('equal')
axes[0].set_title(f'Poincar\u00e9 Disk ($\\alpha$={ALPHA}, visual stretch)\ncolor = radial coord $r$')

# Radial distribution
r_range = np.linspace(0, R, 200)
rho_theory = ALPHA * np.sinh(ALPHA * r_range) / (np.cosh(ALPHA * R) - 1)
axes[1].hist(r, bins=40, density=True, alpha=0.7, color='steelblue', label='Sampled')
axes[1].plot(r_range, rho_theory, 'r-', linewidth=2, label=f'$\\rho(r) = \\alpha\\sinh(\\alpha r)/(\\cosh \\alpha R - 1)$')
axes[1].set_xlabel('Radial coordinate $r$')
axes[1].set_ylabel('Density')
axes[1].set_title(f'Radial Distribution ($\\alpha$={ALPHA})')
axes[1].legend(fontsize=8)

# Angular distribution
axes[2].hist(theta, bins=40, density=True, alpha=0.7, color='steelblue')
axes[2].axhline(1 / (2 * np.pi), color='red', linestyle='--', label='Uniform $1/2\\pi$')
axes[2].set_xlabel('Angular coordinate $\\theta$')
axes[2].set_ylabel('Density')
axes[2].set_title('Angular Distribution')
axes[2].legend()

plt.tight_layout()
plt.show()

## 3. Agent Generation: Gaussian Mixture on $\theta$

Keep the Krioukov radial distribution, but cluster agents angularly.
This pre-seeds community structure while preserving the popularity hierarchy.

In [ ]:
def generate_hyperbolic_gmm(
    n: int,
    R: float,
    n_clusters: int,
    angular_sigma: float,
    rng: np.random.Generator,
    alpha: float = 1.0,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Generate agents with generalized Krioukov radial density + clustered angular positions.
    
    Returns (r, theta, cluster_labels).
    """
    # Radial: generalized Krioukov with alpha
    u = rng.uniform(0, 1, size=n)
    r = np.arccosh(1.0 + u * (np.cosh(alpha * R) - 1.0)) / alpha
    
    # Angular: Gaussian mixture, wrapped to [0, 2pi)
    centers = np.linspace(0, 2 * np.pi, n_clusters, endpoint=False)
    labels = rng.integers(0, n_clusters, size=n)
    theta = centers[labels] + rng.normal(0, angular_sigma, size=n)
    theta = theta % (2 * np.pi)
    
    return r, theta, labels

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
angular_sigmas = [0.15, 0.4, 0.8]

for ax, sigma in zip(axes, angular_sigmas):
    r_g, theta_g, labels_g = generate_hyperbolic_gmm(N, R, 5, sigma, np.random.default_rng(42), alpha=ALPHA)
    x_p, y_p = poincare_coords(r_g, theta_g, R, stretch=0.6)
    
    circle = plt.Circle((0, 0), 1, fill=False, color='black', linewidth=1.5)
    ax.add_patch(circle)
    ax.scatter(x_p, y_p, s=12, alpha=0.6, c=labels_g, cmap='tab10')
    ax.set_xlim(-1.15, 1.15)
    ax.set_ylim(-1.15, 1.15)
    ax.set_aspect('equal')
    ax.set_title(f'$\\sigma_\\theta$ = {sigma}')

plt.suptitle(f'Hyperbolic GMM (5 clusters, N={N}, $\\alpha$={ALPHA})', y=1.02)
plt.tight_layout()
plt.show()

## 4. Tie Formation

### Hard threshold (Eq. 8): connect if $x_{ij} \leq R$
### Soft threshold (Fermi-Dirac): $p(x) = \frac{1}{1 + e^{(x - R) / (2T)}}$

Temperature $T$ controls clustering:
- $T = 0$: hard threshold, maximum clustering
- $T > 0$: softer connections, allows some long-range ties
- $T \to \infty$: random graph (Erdos-Renyi)

In [ ]:
def form_network_hyperbolic_hard(r: np.ndarray, theta: np.ndarray, R: float) -> nx.Graph:
    """Connect all pairs with hyperbolic distance <= R."""
    dist_mat = hyperbolic_distance_matrix(r, theta)
    n = len(r)
    G = nx.Graph()
    G.add_nodes_from(range(n))
    mask = (dist_mat <= R) & (dist_mat > 0)
    edges = list(zip(*np.where(np.triu(mask, k=1))))
    G.add_edges_from(edges)
    return G, dist_mat


def form_network_hyperbolic_soft(
    r: np.ndarray,
    theta: np.ndarray,
    R: float,
    T: float,
    rng: np.random.Generator,
) -> nx.Graph:
    """Connect pairs with Fermi-Dirac probability."""
    dist_mat = hyperbolic_distance_matrix(r, theta)
    n = len(r)
    
    if T <= 0:
        prob = (dist_mat <= R).astype(float)
    else:
        prob = 1.0 / (1.0 + np.exp((dist_mat - R) / (2.0 * T)))
    
    np.fill_diagonal(prob, 0)
    
    # Sample edges from upper triangle
    G = nx.Graph()
    G.add_nodes_from(range(n))
    random_draws = rng.uniform(0, 1, size=(n, n))
    mask = np.triu(random_draws < prob, k=1)
    edges = list(zip(*np.where(mask)))
    G.add_edges_from(edges)
    return G, dist_mat

In [ ]:
# Form network with hard threshold
G_hard, dist_mat = form_network_hyperbolic_hard(r, theta, R)

degrees = [d for _, d in G_hard.degree()]
print(f'Hard threshold network: {G_hard.number_of_edges()} edges, '
      f'mean degree = {np.mean(degrees):.2f}, '
      f'clustering = {nx.average_clustering(G_hard):.4f}')

## 5. Degree Distribution: Is It Heavy-Tailed?

The key sanity check from the Krioukov paper: the degree distribution should follow
a power law $P(k) \sim k^{-\gamma}$ with $\gamma = 2\alpha + 1$ (where $\alpha = 1$ for uniform density gives $\gamma = 3$).

In [ ]:
degrees = np.array([d for _, d in G_hard.degree()])

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Linear histogram
axes[0].hist(degrees, bins=range(0, max(degrees) + 2), density=True, alpha=0.7, color='steelblue')
axes[0].set_xlabel('Degree $k$')
axes[0].set_ylabel('$P(k)$')
axes[0].set_title('Degree Distribution')

# Log-log CCDF (complementary cumulative)
sorted_deg = np.sort(degrees)[::-1]
ccdf = np.arange(1, len(sorted_deg) + 1) / len(sorted_deg)
axes[1].loglog(sorted_deg, ccdf, '.', alpha=0.5, color='steelblue')
# Reference power law: P(K >= k) ~ k^{-(gamma-1)} with gamma=3
k_ref = np.logspace(np.log10(max(1, np.min(degrees[degrees > 0]))), np.log10(np.max(degrees)), 50)
axes[1].loglog(k_ref, (k_ref / k_ref[0]) ** -2 * ccdf[0], 'r--', label='$k^{-2}$ (reference, $\\gamma=3$)')
axes[1].set_xlabel('Degree $k$')
axes[1].set_ylabel('$P(K \\geq k)$')
axes[1].set_title('CCDF (log-log)')
axes[1].legend()

# Degree vs radial position
axes[2].scatter(r, degrees, s=10, alpha=0.5, color='steelblue')
axes[2].set_xlabel('Radial coordinate $r$')
axes[2].set_ylabel('Degree')
axes[2].set_title('Degree vs Position (center = low $r$ = high degree)')

plt.tight_layout()
plt.show()

print(f'Degree stats: min={degrees.min()}, max={degrees.max()}, '
      f'median={np.median(degrees):.0f}, mean={degrees.mean():.1f}')

## 6. Network Visualization

In [ ]:
x_vis, y_vis = poincare_coords(r, theta, R, stretch=0.6)
pos = {i: (x_vis[i], y_vis[i]) for i in range(N)}

fig, ax = plt.subplots(1, 1, figsize=(8, 8))
circle = plt.Circle((0, 0), 1, fill=False, color='black', linewidth=1.5)
ax.add_patch(circle)

# Color nodes by degree
node_colors = [degrees[i] for i in range(N)]
nx.draw_networkx_edges(G_hard, pos, ax=ax, alpha=0.03, width=0.5)
nodes = nx.draw_networkx_nodes(G_hard, pos, ax=ax, node_size=15, node_color=node_colors,
                                cmap='YlOrRd', alpha=0.8)
plt.colorbar(nodes, ax=ax, label='Degree')
ax.set_xlim(-1.15, 1.15)
ax.set_ylim(-1.15, 1.15)
ax.set_aspect('equal')
ax.set_title(f'Hyperbolic Network (N={N}, $\\alpha$={ALPHA}, $\\bar{{k}}$={degrees.mean():.1f})')
plt.tight_layout()
plt.show()

## 7. Structural Hole Metrics at t=0

In [ ]:
def burt_constraint(G: nx.Graph) -> dict[int, float]:
    """Compute Burt's constraint for each node."""
    constraint = {}
    for i in G.nodes():
        neighbors_i = set(G.neighbors(i))
        if len(neighbors_i) == 0:
            constraint[i] = np.nan
            continue
        
        deg_i = len(neighbors_i)
        c_i = 0.0
        for j in neighbors_i:
            p_ij = 1.0 / deg_i
            indirect = 0.0
            for q in neighbors_i:
                if q != j and G.has_edge(q, j):
                    p_iq = 1.0 / deg_i
                    neighbors_q = set(G.neighbors(q))
                    p_qj = 1.0 / len(neighbors_q) if j in neighbors_q else 0.0
                    indirect += p_iq * p_qj
            c_i += (p_ij + indirect) ** 2
        constraint[i] = c_i
    return constraint

In [ ]:
constraint = burt_constraint(G_hard)
betweenness = nx.betweenness_centrality(G_hard)

c_vals = np.array([constraint[i] for i in range(N) if not np.isnan(constraint[i])])
b_vals = np.array([betweenness[i] for i in range(N)])

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Constraint distribution
axes[0, 0].hist(c_vals, bins=40, density=True, alpha=0.7, color='coral')
axes[0, 0].set_xlabel('Constraint (lower = more structural holes)')
axes[0, 0].set_title("Burt's Constraint Distribution")

# Betweenness distribution
axes[0, 1].hist(b_vals, bins=40, density=True, alpha=0.7, color='seagreen')
axes[0, 1].set_xlabel('Betweenness centrality')
axes[0, 1].set_title('Betweenness Centrality Distribution')

# Constraint vs radial position
c_for_scatter = np.array([constraint.get(i, np.nan) for i in range(N)])
valid = ~np.isnan(c_for_scatter)
axes[1, 0].scatter(r[valid], c_for_scatter[valid], s=10, alpha=0.5, color='coral')
axes[1, 0].set_xlabel('Radial coordinate $r$ (low = central)')
axes[1, 0].set_ylabel('Constraint')
axes[1, 0].set_title('Constraint vs Position')

# Betweenness vs radial position
axes[1, 1].scatter(r, b_vals, s=10, alpha=0.5, color='seagreen')
axes[1, 1].set_xlabel('Radial coordinate $r$ (low = central)')
axes[1, 1].set_ylabel('Betweenness')
axes[1, 1].set_title('Betweenness vs Position')

plt.suptitle(f'Structural Hole Metrics (Hyperbolic, N={N})', y=1.02)
plt.tight_layout()
plt.show()

print(f'Constraint: mean={c_vals.mean():.4f}, std={c_vals.std():.4f}')
print(f'Betweenness: mean={b_vals.mean():.4f}, max={b_vals.max():.4f}')

## 8. Temperature Comparison

How does the Fermi-Dirac temperature $T$ affect the initial network?

In [ ]:
temperatures = [0, 0.3, 0.6, 0.9]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))

r_fixed, theta_fixed = generate_hyperbolic_uniform(N, R, np.random.default_rng(42), alpha=ALPHA)
x_vis, y_vis = poincare_coords(r_fixed, theta_fixed, R, stretch=0.6)
pos = {i: (x_vis[i], y_vis[i]) for i in range(N)}

for col, T in enumerate(temperatures):
    if T == 0:
        G, _ = form_network_hyperbolic_hard(r_fixed, theta_fixed, R)
    else:
        G, _ = form_network_hyperbolic_soft(r_fixed, theta_fixed, R, T, np.random.default_rng(42))
    
    degs = np.array([d for _, d in G.degree()])
    
    # Degree distribution
    axes[0, col].hist(degs, bins=range(0, max(degs) + 2), density=True, alpha=0.7, color='steelblue')
    axes[0, col].set_title(f'T={T}: $\\bar{{k}}$={degs.mean():.1f}, CC={nx.average_clustering(G):.3f}')
    axes[0, col].set_xlabel('Degree')
    
    # Network
    circle = plt.Circle((0, 0), 1, fill=False, color='black', linewidth=1)
    axes[1, col].add_patch(circle)
    nx.draw_networkx_edges(G, pos, ax=axes[1, col], alpha=0.02, width=0.5)
    nx.draw_networkx_nodes(G, pos, ax=axes[1, col], node_size=8, node_color='steelblue', alpha=0.5)
    axes[1, col].set_xlim(-1.15, 1.15)
    axes[1, col].set_ylim(-1.15, 1.15)
    axes[1, col].set_aspect('equal')

plt.suptitle(f'Effect of Temperature on Hyperbolic Network (N={N}, $\\alpha$={ALPHA})', y=1.02)
plt.tight_layout()
plt.show()

## Next Steps

- Compare these results directly with the torus initialization notebook
- Investigate the $\alpha$ parameter (non-uniform radial density) for tuning the power-law exponent
- Implement temporal dynamics: attention budgets, tie decay, triadic closure